In [ ]:
from sqlalchemy import create_engine
import pandas as pd
import lzma
import json
import os
from datetime import datetime
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [ ]:
load_dotenv()

DB_NAME=os.getenv('DB_NAME')
DB_USER=os.getenv('DB_USER')
DB_PW=os.getenv('DB_PW')
DB_HOST=os.getenv('DB_HOST')
DB_PORT=os.getenv('DB_PORT')

engine = create_engine(f"postgresql://{DB_USER}:{DB_PW}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
# Scrape bruto do Instagram (~2,1 GB) — NÃO acompanha final/.
# Aponte via INSTAGRAM_SCRAPE_DIR no .env, ou crie o symlink ./raw.
SCRAPE_DIR = os.getenv("INSTAGRAM_SCRAPE_DIR", "raw")


In [ ]:
pastas_scrape = os.listdir(SCRAPE_DIR)
pastas_scrape = [
    os.path.join(SCRAPE_DIR, pasta) for pasta in pastas_scrape
]
pastas_scrape

In [ ]:
arquivos_json = os.listdir('../'+pastas_scrape[2])
timestamp_posts = [x[:x.find('.')] for x in arquivos_json if '.json.xz' in x]
timestamp_posts.sort()
timestamp_posts[:]

In [ ]:
def extrai_dados_post(pasta, nome_arquivo, id_pagina):
    """Retorna link, id e texto"""
    # extrai data
    parte_data = nome_arquivo.replace("_UTC", "") # ex:'2025-04-01_20-58-18_UTC'
    timestamp = datetime.strptime(parte_data, "%Y-%m-%d_%H-%M-%S")

    caminho = os.path.join(pasta, nome_arquivo + '.json')
    caminho_xz = os.path.join(pasta, nome_arquivo + '.json.xz')
    # Retorna o timestamp de modificação (segundos desde 1970)
    timestamp_modificado = os.path.getmtime(caminho_xz)

    # Converte para datetime legível
    data_modificacao = datetime.fromtimestamp(timestamp_modificado)

    # lê json
    with open(caminho, "r", encoding="utf-8") as f:
        dados = json.load(f)
    return {
        'id_publicacao': dados['node']['id'],
        'pagina_id_pagina': id_pagina,
        'data_publicacao': timestamp,
        'data_extracao': data_modificacao,
        'texto_publicacao': dados['node']['caption'],
        'link': dados['node']['shortcode']
    }

In [ ]:
def extrai_dados_comentario(pasta, nome_arquivo, id_pagina):
    caminho = os.path.join(pasta, nome_arquivo + '_comments.json')

    with open(caminho, 'r', encoding='utf-8') as f:
        dados = json.load(f)
    dic_comentarios = {
        'id_comentario': [],
        'publicacao_id_publicacao': [],
        'data_comentario': [],
        'texto_comentario':[]
        }
    for dado in dados:
        dic_comentarios['id_comentario'].append(dado['id'])
        dic_comentarios['publicacao_id_publicacao'].append(id_pagina)
        dic_comentarios['data_comentario'].append(datetime.fromtimestamp(dado['created_at']))
        dic_comentarios['texto_comentario'].append(dado['text'])
    return dic_comentarios
    

In [ ]:
dicionario_pagina = {
    'id_pagina': ['3603220787852024677', '3536902121053814160', '3535996125872183729'],
    'nome_pagina': ['Folha Zona Norte RJ', 'Rio de Nojeira Oficial ®️', 'ZONA OESTE URGENTE'],
    'username': ['folhazonanorte', 'riodenojeiraoficial', 'zonaoesteurgente']
}

df_paginas = pd.DataFrame(dicionario_pagina)
df_paginas

In [ ]:
df_paginas.to_sql('pagina', con=engine, index=False, schema='tcc', if_exists='append')

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
import re

# Conexão com o banco de dados PostgreSQL
conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PW
)
cur = conn.cursor()

# Schema e tabela de destino
schema = "tcc"
tabela = "publicacao"

# Regex para arquivos com timestamp no nome: YYYY-MM-DD_HH-MM-SS.json.xz
padrao_data = re.compile(r"^\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}")

for id in ['3603220787852024677', '3536902121053814160', '3535996125872183729']:

    pasta_atual = os.path.join(SCRAPE_DIR, df_paginas.loc[df_paginas.id_pagina == id, 'username'].values[0])

    # Filtra apenas arquivos com timestamp e termina em .json.xz
    arquivos_json = [
        x for x in os.listdir(pasta_atual)
        if x.endswith(".json.xz") and padrao_data.match(x)
    ]

    # Ordena por timestamp
    timestamp_posts = [x.split(".json.xz")[0] for x in arquivos_json]
    timestamp_posts.sort()

    registros = []
    
    for nome_arquivo in timestamp_posts:
        try:
            dado = extrai_dados_post(pasta_atual, nome_arquivo, id)
            registros.append((
                dado["id_publicacao"],
                dado["pagina_id_pagina"],
                dado["data_publicacao"],
                dado["data_extracao"],
                dado["texto_publicacao"],
                dado["link"]
            ))
        except Exception as e:
            print(f"Erro ao processar {nome_arquivo}: {e}")

    # Inserção em massa no Postgres
    query = f"""
        INSERT INTO {schema}.{tabela} 
        (id_publicacao, pagina_id_pagina, data_publicacao, data_extracao, texto_publicacao, link)
        VALUES %s
        ON CONFLICT (id_publicacao) DO NOTHING;
    """

    if registros:
        execute_values(cur, query, registros)
        conn.commit()
        print(f"Inseridos {len(registros)} registros em {schema}.{tabela}")
    else:
        print(f"Nenhum registro válido encontrado para {id}")

# Fechar conexão
cur.close()
conn.close()


In [ ]:
df_paginas

In [ ]:
conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PW
)
cur = conn.cursor()
tabela = "comentario"


# Regex para nomes de posts no formato: YYYY-MM-DD_HH-MM-SS.json.xz
padrao_data = re.compile(r"^\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}")

for id in ['3603220787852024677', '3536902121053814160', '3535996125872183729']:

    pasta_posts = os.path.join(SCRAPE_DIR, df_paginas.loc[df_paginas.id_pagina == id,'username'].values[0])

    arquivos_json = [
        x for x in os.listdir(pasta_posts)
        if x.endswith(".json.xz") and padrao_data.match(x)
    ]

    # Ordenar posts por timestamp
    timestamp_posts = [x.split(".json.xz")[0] for x in arquivos_json]
    timestamp_posts.sort()

    registros_comentarios = []

    for nome_arquivo in timestamp_posts:
        try:
            # Extrai dados da publicação
            post = extrai_dados_post(pasta_posts, nome_arquivo, id)
            id_publicacao = post["id_publicacao"]

            # Extrai os comentários vinculados a essa publicação
            comentarios = extrai_dados_comentario(pasta_posts, nome_arquivo, id_publicacao)

            for i in range(len(comentarios["id_comentario"])):
                registros_comentarios.append((
                    comentarios["id_comentario"][i],
                    comentarios["publicacao_id_publicacao"][i],
                    comentarios["data_comentario"][i],
                    comentarios["texto_comentario"][i]
                ))

        except Exception as e:
            print(f"[ERRO] Comentários de '{nome_arquivo}': {e}")

    if registros_comentarios:
        query = f"""
            INSERT INTO {schema}.{tabela}
            (id_comentario, publicacao_id_publicacao, data_comentario, texto_comentario)
            VALUES %s
            ON CONFLICT (id_comentario) DO NOTHING;
        """
        execute_values(cur, query, registros_comentarios)
        conn.commit()
        print(f"Inseridos {len(registros_comentarios)} comentários em {schema}.{tabela}")
    else:
        print(f"Nenhum comentário válido para {id}")

cur.close()
conn.close()
